In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("jangedoo/utkface-new")

print("Path to dataset files:", path)

100%|██████████| 331M/331M [00:01<00:00, 178MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/jangedoo/utkface-new/versions/1


In [3]:
import os
import numpy as np
import pandas as pd
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [4]:
folder_path = '/root/.cache/kagglehub/datasets/jangedoo/utkface-new/versions/1/UTKFace'

In [5]:
age=[]
gender=[]
img_path=[]
for file in os.listdir(folder_path):
  age.append(int(file.split('_')[0]))
  gender.append(int(file.split('_')[1]))
  img_path.append(file)

In [6]:
len(age)

23708

In [7]:
df = pd.DataFrame({'age':age,'gender':gender,'img':img_path})

In [8]:
df.shape

(23708, 3)

In [9]:
df.head()

,age,gender,img
0,40,0,40_0_0_20170117203219447.jpg.chip.jpg
1,26,1,26_1_3_20170119193141890.jpg.chip.jpg
2,36,1,36_1_0_20170116165722892.jpg.chip.jpg
3,1,1,1_1_0_20170109194410994.jpg.chip.jpg
4,67,1,67_1_3_20170109132529672.jpg.chip.jpg


In [10]:
train_df = df.sample(frac=1,random_state=0).iloc[:20000]
test_df = df.sample(frac=1,random_state=0).iloc[20000:]

In [11]:
train_df.shape

(20000, 3)

In [12]:
test_df.shape

(3708, 3)

In [13]:
# Data augmentation
train_datagen = ImageDataGenerator(rescale=1./255,
                                   rotation_range=30,
                                   width_shift_range=0.2,
                                   height_shift_range=0.2,
                                   shear_range=0.2,
                                   zoom_range=0.2,
                                   horizontal_flip=True)

test_datagen = ImageDataGenerator(rescale=1./255)

In [14]:
from tensorflow.keras.utils import Sequence

# Custom generator that makes the multi-output data compatible with modern TensorFlow/Keras
class MultiOutputGenerator(Sequence):

    # Initialize the custom generator with the original generator
    def __init__(self, generator):
        self.generator = generator

    # Return the total number of batches in the dataset
    def __len__(self):
        return len(self.generator)

    # Return one batch of images and their corresponding labels
    def __getitem__(self, index):

        # Get images (x) and labels (y) from the original generator
        x, y = self.generator[index]

        # Separate the labels into two outputs:
        # y[:, 0] -> age
        # y[:, 1] -> gender
        # Return them as a tuple to match the model's two outputs
        return x, (y[:, 0], y[:, 1])


In [15]:
# Generators
train_generator_raw = train_datagen.flow_from_dataframe(
    train_df,
    directory=folder_path,
    x_col='img',
    y_col=['age','gender'],
    target_size=(200,200),
    class_mode='raw',
    batch_size=32
)

test_generator_raw = test_datagen.flow_from_dataframe(
    test_df,
    directory=folder_path,
    x_col='img',
    y_col=['age','gender'],
    target_size=(200,200),
    class_mode='raw',
    batch_size=32
)

train_generator = MultiOutputGenerator(train_generator_raw)
test_generator = MultiOutputGenerator(test_generator_raw)


Found 20000 validated image filenames.
Found 3708 validated image filenames.


In [16]:
from keras.applications.resnet50 import ResNet50
from keras.layers import *
from keras.models import Model

In [17]:
resnet = ResNet50(include_top=False, input_shape=(200,200,3))

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [18]:
# Transfer Learning
resnet.trainable=False

output = resnet.layers[-1].output

flatten = Flatten()(output)

dense1 = Dense(512, activation='relu')(flatten)
dense2 = Dense(512,activation='relu')(flatten)

dense3 = Dense(512,activation='relu')(dense1)
dense4 = Dense(512,activation='relu')(dense2)

output1 = Dense(1,activation='linear',name='age')(dense3)
output2 = Dense(1,activation='sigmoid',name='gender')(dense4)

In [19]:
model = Model(inputs=resnet.input,outputs=[output1,output2])

In [20]:
model.compile(optimizer='adam', loss={'age': 'mae', 'gender': 'binary_crossentropy'}, metrics={'age': 'mae', 'gender': 'accuracy'},loss_weights={'age':1,'gender':99})

In [22]:
model.fit(train_generator, epochs=10, validation_data=test_generator)

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 227s 364ms/step - age_loss: 14.4177 - age_mae: 14.4177 - gender_accuracy: 0.5224 - gender_loss: 0.6923 - loss: 82.9520 - val_age_loss: 13.7769 - val_age_mae: 13.7769 - val_gender_accuracy: 0.5240 - val_gender_loss: 0.6920 - val_loss: 82.2870
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 237s 380ms/step - age_loss: 14.4066 - age_mae: 14.4066 - gender_accuracy: 0.5224 - gender_loss: 0.6922 - loss: 82.9324 - val_age_loss: 13.5904 - val_age_mae: 13.5904 - val_gender_accuracy: 0.5240 - val_gender_loss: 0.6921 - val_loss: 82.1038
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 225s 360ms/step - age_loss: 14.3340 - age_mae: 14.3340 - gender_accuracy: 0.5224 - gender_loss: 0.6922 - loss: 82.8637 - val_age_loss: 13.5018 - val_age_mae: 13.5018 - val_gender_accuracy: 0.5240 - val_gender_loss: 0.6920 - val_loss: 82.0093
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 225s 359ms/step - age_loss: 14.2514 - age_mae: 14.2514 - gender_accuracy: 0.5228 - gender_loss: 0.6928 - loss: 82.